### Baseline Models

[x] Logistic regression

[x] SVM

[x] Random Forests (for baselining)

[x] XGBoost

[x] K-NN

[x] Naive Bayes

[-] Decision trees

[-] Neural Networks

In [1]:
import sys
from pathlib import Path

# cwd can be repo root, src/, or src/feature_selection/ — walk up until src/stroke_data.py exists
_here = Path().resolve()
REPO_ROOT = _here
while REPO_ROOT != REPO_ROOT.parent:
    if (REPO_ROOT / "src" / "stroke_data.py").is_file():
        break
    REPO_ROOT = REPO_ROOT.parent
else:
    raise FileNotFoundError("Could not find src/stroke_data.py (open this project from the repo folder).")

sys.path.insert(0, str(REPO_ROOT / "src"))

In [2]:
import sklearn
import scipy
import numpy as np
from stroke_data import get_stroke_data_for_cv, get_stroke_data

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

from sklearn.model_selection import GridSearchCV

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

In [3]:
X_train, X_test, y_train, y_test = get_stroke_data_for_cv("data/knn-standardize-distance.csv")

#### Logistic Regression

https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html

In [8]:
param_lr = {"C": [0.001, 0.01, 0.1, 1.0, 10.0, 100, 1000],
            "solver": ["liblinear", "newton-cg", "newton-cholesky", "sag", "saga"]
            }

lr = LogisticRegression(random_state=42)
grid_search_lr = GridSearchCV(estimator=lr, param_grid=param_lr, scoring="f1") # , verbose=5)
grid_search_lr.fit(X=X_train, y=y_train)

best_lr_model = grid_search_lr.best_estimator_
print("Best params = ", grid_search_lr.best_params_)

best_lr_model.fit(X=X_train, y=y_train)

lr_preds_train = best_lr_model.predict(X_train)
lr_preds = best_lr_model.predict(X_test)

print("-- Train --")
print("Accuracy = ", accuracy_score(y_train, lr_preds_train))
print("F1 = ", f1_score(y_train, lr_preds_train))
print("Precision = ", precision_score(y_train, lr_preds_train))
print("Recall = ", recall_score(y_train, lr_preds_train))

print("-- Test --")
print("Accuracy = ", accuracy_score(y_test, lr_preds))
print("F1 = ", f1_score(y_test, lr_preds))
print("Precision = ", precision_score(y_test, lr_preds))
print("Recall = ", recall_score(y_test, lr_preds))

Best params =  {'C': 0.001, 'solver': 'liblinear'}
-- Train --
Accuracy =  0.951320939334638
F1 =  0.0
Precision =  0.0
Recall =  0.0
-- Test --
Accuracy =  0.9510763209393346
F1 =  0.0
Precision =  0.0
Recall =  0.0


/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


### SVM 

https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html

In [9]:
param_svm = {"C": [0.001, 0.01, 0.1, 1.0, 10.0],
             "kernel": ["linear", "rbf"], 
             "gamma": [0.01, 0.1, 1, 10, 100, "auto", "scale"],
             }

svm = SVC()
grid_search_svm = GridSearchCV(estimator=svm, param_grid=param_svm, scoring="f1") # , verbose=5)
grid_search_svm.fit(X=X_train, y=y_train)

best_svm_model = grid_search_svm.best_estimator_
print("Best params = ", grid_search_svm.best_params_)

best_svm_model.fit(X=X_train, y=y_train)

svm_preds_train = best_svm_model.predict(X_train)
svm_preds = best_svm_model.predict(X_test)

print("-- Train --")
print("Accuracy = ", accuracy_score(y_train, svm_preds_train))
print("F1 = ", f1_score(y_train, svm_preds_train))
print("Precision = ", precision_score(y_train, svm_preds_train))
print("Recall = ", recall_score(y_train, svm_preds_train))

print("-- Test --")
print("Accuracy = ", accuracy_score(y_test, svm_preds))
print("F1 = ", f1_score(y_test, svm_preds))
print("Precision = ", precision_score(y_test, svm_preds))
print("Recall = ", recall_score(y_test, svm_preds))

Best params =  {'C': 10.0, 'gamma': 1, 'kernel': 'rbf'}
-- Train --
Accuracy =  0.9877690802348337
F1 =  0.8579545454545454
Precision =  0.9869281045751634
Recall =  0.7587939698492462
-- Test --
Accuracy =  0.9227005870841487
F1 =  0.04819277108433735
Precision =  0.06060606060606061
Recall =  0.04


### Random Forests 
https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html

In [10]:
param_rf = {"n_estimators": [5, 10, 15, 20],
            "max_features": ["sqrt", "log2", None],
            "max_depth": [5, 10, 15, None],
            "max_leaf_nodes": [5, 10, 15, None],
            "bootstrap": [True, False] 
            }

rf = RandomForestClassifier()
grid_search_rf = GridSearchCV(estimator=rf, param_grid=param_rf, scoring="f1") # , verbose=5)
grid_search_rf.fit(X=X_train, y=y_train)

best_rf_model = grid_search_rf.best_estimator_
print("Best params = ", grid_search_rf.best_params_)

best_rf_model.fit(X=X_train, y=y_train)

rf_preds_train = best_rf_model.predict(X_train)
rf_preds = best_rf_model.predict(X_test)

print("-- Train --")
print("Accuracy = ", accuracy_score(y_train, rf_preds_train))
print("F1 = ", f1_score(y_train, rf_preds_train))
print("Precision = ", precision_score(y_train, rf_preds_train))
print("Recall = ", recall_score(y_train, rf_preds_train))

print("-- Test --")
print("Accuracy = ", accuracy_score(y_test, rf_preds))
print("F1 = ", f1_score(y_test, rf_preds))
print("Precision = ", precision_score(y_test, rf_preds))
print("Recall = ", recall_score(y_test, rf_preds))

Best params =  {'bootstrap': False, 'max_depth': None, 'max_features': None, 'max_leaf_nodes': None, 'n_estimators': 15}
-- Train --
Accuracy =  1.0
F1 =  1.0
Precision =  1.0
Recall =  1.0
-- Test --
Accuracy =  0.910958904109589
F1 =  0.11650485436893204
Precision =  0.11320754716981132
Recall =  0.12


### XGBoost

https://xgboost.readthedocs.io/en/latest/parameter.html

https://xgboost.readthedocs.io/en/latest/python/sklearn_estimator.html

In [11]:
param_xgb = {
            "max_depth": [6, 10, 15, 20],
            "subsample": [0.1, 0.5, 1], # subsampling helps prevent overfitting. High subsampling number=high overfitting change
            "lambda": [0.5],
            "gamma": [0.5, 1 ,2],
            "objective": ["binary:logistic"],
            "eta": [0.1, 0.3, 1],
            }

xgb = XGBClassifier()
grid_search_xgb = GridSearchCV(estimator=xgb, param_grid=param_xgb, scoring="f1")#, verbose=3)
grid_search_xgb.fit(X=X_train, y=y_train)

best_xgb_model = grid_search_xgb.best_estimator_
print("Best params = ", grid_search_xgb.best_params_)

best_xgb_model.fit(X=X_train, y=y_train)


train_xgb_preds = best_xgb_model.predict(X_train)
xgb_preds = best_xgb_model.predict(X_test)

print("-- Train --")
print("Accuracy = ", accuracy_score(y_train, train_xgb_preds))
print("F1 = ", f1_score(y_train, train_xgb_preds))
print("Precision = ", precision_score(y_train, train_xgb_preds))
print("Recall = ", recall_score(y_train, train_xgb_preds))

print("-- Test --")
print("Accuracy = ", accuracy_score(y_test, xgb_preds))
print("F1 = ", f1_score(y_test, xgb_preds))
print("Precision = ", precision_score(y_test, xgb_preds))
print("Recall = ", recall_score(y_test, xgb_preds))

Best params =  {'eta': 1, 'gamma': 2, 'lambda': 0.5, 'max_depth': 6, 'objective': 'binary:logistic', 'subsample': 0.1}
-- Train --
Accuracy =  0.9246575342465754
F1 =  0.23383084577114427
Precision =  0.2315270935960591
Recall =  0.23618090452261306
-- Test --
Accuracy =  0.9246575342465754
F1 =  0.18947368421052632
Precision =  0.2
Recall =  0.18


### Naive Bayes

https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.GaussianNB.html#sklearn.naive_bayes.GaussianNB

In [16]:
param_nb = {
            "var_smoothing": [1e-11, 1e-10, 1e-9, 1e-8, 1e-7]   # default is 1e-9
            }

nb = GaussianNB()
grid_search_nb = GridSearchCV(estimator=nb, param_grid=param_nb, scoring="f1")#, verbose=3)
grid_search_nb.fit(X=X_train, y=y_train)

best_nb_model = grid_search_nb.best_estimator_
print("Best params = ", grid_search_nb.best_params_)

best_nb_model.fit(X=X_train, y=y_train)

train_nb_preds = best_nb_model.predict(X_train)
nb_preds = best_nb_model.predict(X_test)

print("-- Train --")
print("Accuracy = ", accuracy_score(y_train, train_nb_preds))
print("F1 = ", f1_score(y_train, train_nb_preds))
print("Precision = ", precision_score(y_train, train_nb_preds))
print("Recall = ", recall_score(y_train, train_nb_preds))

print("-- Test --")
print("Accuracy = ", accuracy_score(y_test, nb_preds))
print("F1 = ", f1_score(y_test, nb_preds))
print("Precision = ", precision_score(y_test, nb_preds))
print("Recall = ", recall_score(y_test, nb_preds))

Best params =  {'var_smoothing': 1e-11}
-- Train --
Accuracy =  0.8610567514677103
F1 =  0.22826086956521738
Precision =  0.1564245810055866
Recall =  0.4221105527638191
-- Test --
Accuracy =  0.8639921722113503
F1 =  0.24043715846994534
Precision =  0.16541353383458646
Recall =  0.44


### KNN

https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html

In [17]:
param_knn = {
    "n_neighbors": [1, 2, 3, 5, 10]
}

knn = KNeighborsClassifier()
grid_search_knn = GridSearchCV(estimator=knn, param_grid=param_knn, scoring="f1") #, verbose=5)
grid_search_knn.fit(X=X_train, y=y_train)

best_knn_model = grid_search_knn.best_estimator_
print("Best params = ", grid_search_knn.best_params_)

best_knn_model.fit(X=X_train, y=y_train)

train_knn_preds = best_knn_model.predict(X_train)
knn_preds = best_knn_model.predict(X_test)

print("-- Train --")
print("Accuracy = ", accuracy_score(y_train, train_knn_preds))
print("F1 = ", f1_score(y_train, train_knn_preds))
print("Precision = ", precision_score(y_train, train_knn_preds))
print("Recall = ", recall_score(y_train, train_knn_preds))

print("-- Test --")
print("Accuracy = ", accuracy_score(y_test, knn_preds))
print("F1 = ", f1_score(y_test, knn_preds))
print("Precision = ", precision_score(y_test, knn_preds))
print("Recall = ", recall_score(y_test, knn_preds))

Best params =  {'n_neighbors': 1}
-- Train --
Accuracy =  1.0
F1 =  1.0
Precision =  1.0
Recall =  1.0
-- Test --
Accuracy =  0.9090019569471625
F1 =  0.041237113402061855
Precision =  0.0425531914893617
Recall =  0.04


In [18]:
best_knn_model.get_params()

{'algorithm': 'auto',
 'leaf_size': 30,
 'metric': 'minkowski',
 'metric_params': None,
 'n_jobs': None,
 'n_neighbors': 1,
 'p': 2,
 'weights': 'uniform'}

### Neural Networks

https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html#sklearn.neural_network.MLPClassifier

In [4]:
param_mlp = {"hidden_layer_sizes": [(100, 100, 100), (100, 100, 100, 100), (100, 100, 100, 100, 100)],
             "solver": ["adam", "sgd"],
             "alpha": [0.0001, 0.001, 0.01, 1.0],
             "max_iter": [200, 500, 1000, 1500]
            }

mlp = MLPClassifier(random_state=42)
grid_search_mlp = GridSearchCV(estimator=mlp, param_grid=param_mlp, scoring="f1")
grid_search_mlp.fit(X=X_train, y=y_train)

best_mlp_model = grid_search_mlp.best_estimator_
print("Best params = ", grid_search_mlp.best_params_)

best_mlp_model.fit(X=X_train, y=y_train)

train_mlp_preds = best_mlp_model.predict(X_train)
mlp_preds = best_mlp_model.predict(X_test)

print("-- Train --")
print("Accuracy = ", accuracy_score(y_train, train_mlp_preds))
print("F1 = ", f1_score(y_train, train_mlp_preds))
print("Precision = ", precision_score(y_train, train_mlp_preds))
print("Recall = ", recall_score(y_train, train_mlp_preds))

print("-- Test --")
print("Accuracy = ", accuracy_score(y_test, mlp_preds))
print("F1 = ", f1_score(y_test, mlp_preds))
print("Precision = ", precision_score(y_test, mlp_preds))
print("Recall = ", recall_score(y_test, mlp_preds))

/opt/anaconda3/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/neural_network/_

Best params =  {'alpha': 0.001, 'hidden_layer_sizes': (100, 100, 100, 100, 100), 'max_iter': 200, 'solver': 'adam'}
-- Train --
Accuracy =  0.9958414872798435
F1 =  0.9578163771712159
Precision =  0.946078431372549
Recall =  0.9698492462311558
-- Test --
Accuracy =  0.9256360078277887
F1 =  0.13636363636363635
Precision =  0.15789473684210525
Recall =  0.12
